# 01 — Generate Synthetic SA Banking Data

This notebook generates realistic South African retail banking transaction data from scratch.

We generate:
- **10,000 customers** across 4 SA segments (salaried professional, entry-level, informal income, SASSA recipients)
- **~500,000 transactions** over 24 months (Jan 2023 – Dec 2024)
- SA merchants: Shoprite, Checkers, Vodacom, Engen, DStv, Capitec, and more
- Realistic SA pay patterns: salary on the 25th, SASSA grants on the 1st, month-end spending squeeze

In [ ]:
# Install dependencies if not already available
%pip install faker pandas numpy -q

In [ ]:
import sys
sys.path.insert(0, '/Workspace/Repos/sa-banking-behaviour-pipeline')

# If running from DBFS, use this path instead:
# sys.path.insert(0, '/dbfs/FileStore/sa-banking-behaviour-pipeline')

## Step 1 — Import the generator

The `transaction_generator.py` module lives in `src/generator/`. It contains all the logic for creating realistic SA customer profiles and spending behaviour.

In [ ]:
from src.generator.transaction_generator import generate_customers, generate_transactions
from datetime import datetime
import pandas as pd

print('Generator imported successfully')

## Step 2 — Generate customers

We create 10,000 customers split across 4 segments:
- **Salaried professional** (35%) — R18k–R80k/month, high spending
- **Salaried entry level** (30%) — R5k–R18k/month, medium spending
- **Informal income** (20%) — R2k–R8k/month, irregular deposits
- **SASSA recipient** (15%) — R350–R2,090/month, very low spending

In [ ]:
print('Generating customers...')
customers_df = generate_customers(n=10000)
print(f'Generated: {len(customers_df):,} customers')
customers_df.head(10)

In [ ]:
# Check segment distribution
customers_df['segment'].value_counts(normalize=True).round(3) * 100

## Step 3 — Generate transactions

For each customer we generate transactions across 24 months. The number of transactions per month depends on the customer segment — professionals transact more frequently than SASSA recipients.

In [ ]:
print('Generating transactions (this takes 2-3 minutes)...')
start = datetime(2023, 1, 1)
end = datetime(2024, 12, 31)

transactions_df = generate_transactions(customers_df, start, end)
print(f'Generated: {len(transactions_df):,} transactions')
transactions_df.head(10)

## Step 4 — Quick sanity checks

Before saving, we verify the data looks realistic.

In [ ]:
print('=== Transaction Type Split ===')
print(transactions_df['transaction_type'].value_counts())

print('\n=== Top 10 Merchants ===')
print(transactions_df[transactions_df['transaction_type']=='DEBIT']['merchant_name'].value_counts().head(10))

print('\n=== Amount Stats (ZAR) ===')
print(transactions_df['amount'].describe().round(2))

In [ ]:
# Check SA pay cycle — salary deposits should spike on the 25th
import matplotlib.pyplot as plt

credits = transactions_df[transactions_df['transaction_type'] == 'CREDIT'].copy()
credits['day'] = pd.to_datetime(credits['transaction_date']).dt.day

plt.figure(figsize=(14, 4))
credits['day'].value_counts().sort_index().plot(kind='bar', color='#1a73e8')
plt.title('Credit Deposits by Day of Month — SA Pay Cycle Visible on Day 25')
plt.xlabel('Day of Month')
plt.ylabel('Number of Deposits')
plt.tight_layout()
plt.show()

## Step 5 — Save to DBFS

We save both CSVs to Databricks File System (DBFS) so the next notebook can pick them up.

In [ ]:
# Save to DBFS
DBFS_PATH = '/dbfs/FileStore/sa-banking-pipeline/raw'
import os
os.makedirs(DBFS_PATH, exist_ok=True)

customers_df.to_csv(f'{DBFS_PATH}/customers.csv', index=False)
transactions_df.to_csv(f'{DBFS_PATH}/transactions.csv', index=False)

print(f'customers.csv saved: {len(customers_df):,} rows')
print(f'transactions.csv saved: {len(transactions_df):,} rows')
print(f'Path: {DBFS_PATH}')

## ✅ Done

Raw data is now sitting in DBFS ready for Bronze ingestion.

**Next:** Open `02_bronze_ingestion.ipynb`